In [6]:
from datasets import load_dataset, concatenate_datasets, interleave_datasets

# 1. Load the Datasets
mass_ds = load_dataset("Ganesh01kumar02reddy/medical-reasoning-processed", split="train")
brain_ds = load_dataset("akash2402/OpenMd-medical-reasoning-dataset-lite", split="train")

# 2. ALIGNMENT for mass_ds (Using the names from your Error Message)
# We will treat 'user_content' as the instruction 
# and combine 'reasoning_content' + 'assistant_content' as the output
def format_mass(example):
    return {
        "instruction": example["user_content"],
        "output": f"REASONING: {example['reasoning_content']}\n\nRESPONSE: {example['assistant_content']}"
    }

mass_ds = mass_ds.map(format_mass, remove_columns=mass_ds.column_names)

# 3. ALIGNMENT for brain_ds
# Let's check columns for brain_ds (usually 'instruction' and 'output')
# If they differ, we map them here:
def format_brain(example):
    # Adjust 'instruction' and 'output' keys below if brain_ds uses different names
    return {
        "instruction": example.get("instruction", example.get("input", "")),
        "output": example.get("output", example.get("reasoning", ""))
    }

brain_ds = brain_ds.map(format_brain, remove_columns=brain_ds.column_names)

# 4. The Strategic Blend
mixed_ds = interleave_datasets(
    [mass_ds, brain_ds], 
    probabilities=[0.95, 0.05], 
    stopping_strategy="all_exhausted"
)

# 5. Save to Parquet (HPC Optimized)
mixed_ds.to_parquet("medical_reasoning_combined.parquet")

print(f"✅ Fixed! Total rows: {len(mixed_ds)}")
print(f"📊 New Column Names: {mixed_ds.column_names}")

README.md:   0%|          | 0.00/403 [00:00<?, ?B/s]

data/train-00000-of-00009.parquet:   0%|          | 0.00/274M [00:00<?, ?B/s]

data/train-00001-of-00009.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

data/train-00002-of-00009.parquet:   0%|          | 0.00/274M [00:00<?, ?B/s]

data/train-00003-of-00009.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/501 [00:00<?, ? examples/s]

Map:   0%|          | 0/506150 [00:00<?, ? examples/s]

Map:   0%|          | 0/18999 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/44 [00:00<?, ?ba/s]

✅ Fixed! Total rows: 532740
📊 New Column Names: ['instruction', 'output']


In [1]:
# 1. Install pre-compiled xformers for CUDA 12.1 (Standard for Kaggle 2026)
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu121

# 2. Install Unsloth and other dependencies without re-building xformers
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 4.0 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 41.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 80.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 12.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 2.5 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Force-reinstall to sync unsloth and unsloth_zoo
!pip install --upgrade --no-deps --force-reinstall unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.2 MB/s eta 0:00:00
  Using cached unsloth_zoo-2026.4.9-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 28.5 MB/s eta 0:00:00:00:0100:01
Using cached unsloth_zoo-2026.4.9-py3-none-any.whl (421 kB)
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.4.9
    Uninstalling unsloth_zoo-2026.4.9:
      Successfully uninstalled unsloth_zoo-2026.4.9
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.4.8
    Uninstalling unsloth-2026.4.8:
      Successfully uninstalled unsloth-2026.4.8


In [15]:
# 1. Clear the GPU memory first
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

# 2. Re-tokenize with a smaller sequence length (512 is plenty for a test)
inputs = tokenizer(
    prompts,
    truncation = True,
    max_length = 512, # <--- Reduced from 2048 to save 75% memory
    padding = "max_length",
    return_tensors = "pt",
).to("cuda")

# 3. Enable Gradient Checkpointing (Saves VRAM by recalculating math)
model.gradient_checkpointing_enable()

# 4. Try the Forward Pass again
model.train()
with torch.cuda.amp.autocast(): # Uses mixed precision for even more savings
    outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss

loss.backward()

print(f"MANUAL SMOKE TEST SUCCESS!")
print(f"Current Loss: {loss.item():.4f}")

/tmp/ipykernel_57/230974769.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # Uses mixed precision for even more savings


Unsloth: Will smartly offload gradients to save VRAM!
🔥 MANUAL SMOKE TEST SUCCESS!
Current Loss: 3.2988
